In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

# Load dataset
# The '..' navigates up one directory from 'notebooks' to access 'data'
df = pd.read_csv('../data/fertilizer_recommendation.csv')

# Display the first 5 rows to verify it loaded correctly
df.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Nitrogen_Level,Phosphorus_Level,Potassium_Level,Temperature,Humidity,Rainfall,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Previous_Crop,Region,Fertilizer_Used_Last_Season,Yield_Last_Season,Recommended_Fertilizer
0,Clay,6.07,34.98,0.32,1.87,61,44,84,19.84,83.31,1693.22,Cotton,Harvest,Kharif,Canal,Wheat,South,297.15,1.19,MOP
1,Silt,6.39,47.34,0.28,0.21,59,56,18,24.40,46.27,1030.21,Maize,Vegetative,Kharif,Sprinkler,Potato,Central,77.17,4.40,Urea
2,Sandy,7.92,38.13,0.99,1.88,43,21,119,24.82,71.86,1166.16,Cotton,Flowering,Kharif,Rainfed,Tomato,South,128.93,7.21,Urea
3,Clay,5.86,14.17,1.46,0.36,88,46,34,27.87,53.23,2881.83,Wheat,Flowering,Zaid,Sprinkler,Potato,West,233.96,1.85,MOP
4,Clay,7.98,19.28,0.85,2.16,104,53,98,24.17,51.87,714.84,Potato,Sowing,Kharif,Rainfed,Maize,East,214.39,7.36,Zinc Sulphate


In [2]:
# Check for missing values in every column
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values[missing_values > 0])

# Note: If your dataset is clean, this will return an empty Series. If there are missing values, you can decide how to handle them (e.g., imputation, dropping rows/columns).

Missing Values:
 Series([], dtype: int64)


In [3]:
# 'Recommended_Fertilizer' is our target variable
X = df.drop('Recommended_Fertilizer', axis=1)
y = df['Recommended_Fertilizer']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

Features shape: (10000, 19)
Target shape: (10000,)


In [4]:
# Identify categorical and numerical columns dynamically
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f"Categorical features: {len(categorical_cols)}")
print(f"Numerical features: {len(numerical_cols)}")

# Build the preprocessor pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        # handle_unknown='ignore' ensures the app won't crash if a user inputs a brand new crop type later
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols) 
    ])

# Fit and transform the features
X_processed = preprocessor.fit_transform(X)

# We must also encode our target variable (the fertilizer names) into numbers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

Categorical features: 7
Numerical features: 12


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, 
    y_encoded, 
    test_size=0.2, 
    random_state=42, # Setting a seed ensures reproducibility
    stratify=y_encoded # Ensures proportional representation of all fertilizer types in both sets
)

print(f"Training data size: {X_train.shape}")
print(f"Testing data size: {X_test.shape}")

Training data size: (8000, 46)
Testing data size: (2000, 46)


In [6]:
# Save the preprocessor and label encoder to the 'models' directory
joblib.dump(preprocessor, '../models/preprocessor.joblib')
joblib.dump(label_encoder, '../models/label_encoder.joblib')

# We can also save our processed data arrays to quickly load them in the training phase
np.save('../data/X_train.npy', X_train)
np.save('../data/X_test.npy', X_test)
np.save('../data/y_train.npy', y_train)
np.save('../data/y_test.npy', y_test)

print("Preprocessing complete. Assets saved successfully!")

Preprocessing complete. Assets saved successfully!
